# 00 · The project in one notebook

**Execution status:** 🟢 EXECUTED (CPU, outputs saved)
**Plan reference:** PLAN Part 0 (executive summary) · Part V.3 (notebook map)

This is the map for the whole study. If you read one notebook, read this one: it states the question, the two stages, the *correctness oracle* that every claim must clear, the eight research questions with their decision rules, and how the eleven notebooks fit together. Every fact here is pulled live from the frozen configs, so the map cannot drift from the territory.

| | |
|---|---|
| **Inputs** | `config/*.yaml`, `PLAN.md`, `PREREGISTRATION.md` |
| **Outputs** | the RQ table, the dependency graph, the notebook↔figure map, current status |
| **Runtime** | < 10 s |

*Project 19 — Anatomy of a Design Skill. Governance: `CLAUDE.md`. Plan: `PLAN.md`. Derivations:
`THEORY.md`. Freeze: `PREREGISTRATION.md`. This notebook imports tested machinery from the `p19`
package and carries the narrative; it never re-implements logic that lives in `src/p19/`.*

In [1]:
%matplotlib inline
# Bootstrap: locate the repo root (repo-relative — no hardcoded paths) and make p19 importable.
import sys, pathlib
_here = pathlib.Path.cwd()
_root = next((c for c in [_here, *_here.parents]
              if (c / "pyproject.toml").exists() and (c / "src" / "p19").exists()), _here)
if str(_root / "src") not in sys.path:
    sys.path.insert(0, str(_root / "src"))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from p19 import REPO_ROOT
np.random.seed(0)                      # notebook-level seed; every generator also takes an explicit seed
pd.set_option("display.max_columns", 40); pd.set_option("display.width", 120)
print("p19 ready · repo:", REPO_ROOT.name)

p19 ready · repo: steering-llm-aesthetics


## 0.1 · The question, in one paragraph

LLMs produce better user interfaces when you hand them a short *frontend-design skill* — a paragraph
or two of guidance. Everyone reports the effect; nobody has measured it. We measure it in **two
committed stages on one open-weights model**, `Qwen/Qwen2.5-Coder-7B-Instruct` (locked, ADR-001), so
that *"which instruction matters"* (Stage 1) and *"where it lives inside the model"* (Stage 2)
describe the **same system**.

- **Stage 1 — black box.** A factorial ablation of a five-component design skill: which components are
  *necessary*, which are *sufficient*, and how they *interact*.
- **Stage 2 — white box.** Mechanistic interpretability: locate the skill's signal in the residual
  stream, extract a diff-in-means *"clean-design direction,"* and **causally verify** it by steering
  the model with **no design prompt** on held-out prompts — including task types never seen in
  development.

## 0.2 · The correctness oracle (non-negotiable)

Every quality claim — an ablation delta **or** a steering delta — must move **two independent
signals**:

1. **Objective metrics on the *rendered* output** (headless Chromium, rendered DOM + screenshot —
   never raw code strings): render-success, WCAG contrast, axe violations, layout regularity, spacing,
   overflow/overlap, palette statistics, and a validated **purple-slop index**. Deterministic and
   unfoolable. Summarized into one **Primary Objective Composite (POC)**.
2. **A pairwise preference protocol** (VLM judge, human-validated) gated behind chance-corrected
   inter-rater agreement **and** a power check, analyzed with a **style-controlled Bradley–Terry**
   model.

> **The null rule (verbatim, PREREG §8).** *An ablated component's effect, or a steering direction's
> effect, is reported as **null** unless it moves an objective-metric endpoint OR a preference delta
> that passes both the reliability gate and the power gate. Aesthetic claims not backed by the
> objective half are inadmissible.* **Stage-2 bar:** a "design direction" is real only if steering it
> with **no design prompt** causally reproduces a measurable quality gain on held-out prompts.

## 0.3 · The eight research questions

Each RQ carries a hypothesis, the two-signal measurement, and a **preregistered decision rule** with
an explicit **honest-null branch** (a clean null is a first-class result). The table is the study's
spine — the analysis notebooks (05, 07a, 08) render each decision against data.

In [2]:
# The eight RQs with their hypotheses, endpoints, and preregistered decision rules (PLAN Part I).
import pandas as pd
rq = pd.DataFrame([
 ["RQ1","Necessity","removing Cᵢ from FULL lowers quality (LOO)","POC + BT, Holm(12)","Cᵢ necessary if δ>0 & Holm-sig OR gated BT CI>0; rank by δ","necessary(ranked) / null / negative"],
 ["RQ2","Sufficiency","Cᵢ alone lifts above NEUTRAL (AOI); none reaches FULL","POC + BT, Holm(12)","sufficient-partial if δ>0 & sig; distributed if none ≥30% of gap","one-carries / distributed / null"],
 ["RQ3","Interactions","≥1 two-factor interaction is non-zero","2FI coefs, BH(10)","pair interacts if BH-sig & concordant on ≥1 family","additive / interacting(pairs)"],
 ["RQ4","5 words vs 391 tokens","FULL > BEAUTY1 > NEUTRAL","ordered POC+BT","content if FULL−BEAUTY1>0; nudge if BEAUTY1≥70% gap","content / nudge / neither"],
 ["RQ5","Locate","signal is linearly decodable in a mid-band (≈11–20)","probe AUC+selectivity, patching","band = contiguous AUC≥.80 & sel≥.15 & patch≥25%","localized / diffuse"],
 ["RQ6","Steer (the bar)","diff-in-means, no prompt, raises held-out quality","4-arm, both signals","§7 success criterion (Holm + gated BT + random control + flip)","sufficient(&/or necessary) / null"],
 ["RQ7","Failure surface","generalizes OOD, nonzero anti-steerable, non-monotone","per-type deltas, BH","generalizes if unseen delta>0 & anti-steer<50%","robust / partial / brittle"],
 ["RQ8","Correspondence","component sub-vectors near-orthogonal, each steers its family","cosine + signature match","found if mean|cos|<.3 & ≥3/5 signatures match","found / partial / absent"],
], columns=["RQ","name","hypothesis","endpoints","decision rule","outcomes"])
rq.set_index("RQ")

,name,hypothesis,endpoints,decision rule,outcomes
RQ,,,,,
RQ1,Necessity,removing Cᵢ from FULL lowers quality (LOO),"POC + BT, Holm(12)",Cᵢ necessary if δ>0 & Holm-sig OR gated BT CI>...,necessary(ranked) / null / negative
RQ2,Sufficiency,Cᵢ alone lifts above NEUTRAL (AOI); none reach...,"POC + BT, Holm(12)",sufficient-partial if δ>0 & sig; distributed i...,one-carries / distributed / null
RQ3,Interactions,≥1 two-factor interaction is non-zero,"2FI coefs, BH(10)",pair interacts if BH-sig & concordant on ≥1 fa...,additive / interacting(pairs)
RQ4,5 words vs 391 tokens,FULL > BEAUTY1 > NEUTRAL,ordered POC+BT,content if FULL−BEAUTY1>0; nudge if BEAUTY1≥70...,content / nudge / neither
RQ5,Locate,signal is linearly decodable in a mid-band (≈1...,"probe AUC+selectivity, patching",band = contiguous AUC≥.80 & sel≥.15 & patch≥25%,localized / diffuse
RQ6,Steer (the bar),"diff-in-means, no prompt, raises held-out quality","4-arm, both signals",§7 success criterion (Holm + gated BT + random...,sufficient(&/or necessary) / null
RQ7,Failure surface,"generalizes OOD, nonzero anti-steerable, non-m...","per-type deltas, BH",generalizes if unseen delta>0 & anti-steer<50%,robust / partial / brittle
RQ8,Correspondence,"component sub-vectors near-orthogonal, each st...",cosine + signature match,found if mean|cos|<.3 & ≥3/5 signatures match,found / partial / absent


**Reading the table.** RQ1–RQ4 are Stage 1 (black-box ablation, notebook 05). RQ5–RQ8 are Stage 2
(white-box interpretability, notebooks 06–07). The two **confirmatory** families use **Holm** (FWER
0.05); everything exploratory uses **Benjamini–Hochberg** (FDR 0.10). The priors we walk in with:
**C5 (negatives)** and **C1 (color)** should punch above their token weight (RQ1); sufficiency should
be *distributed* (RQ2); a mid-layer direction should steer *partially* (RQ6) with an OOD failure tail
(RQ7).

## 0.4 · The dependency graph — what blocks what

The project has a strict phase order. The **CPU-now** work (this environment) is complete and green;
the freeze tag then unlocks the **GPU phase** (Colab). Everything below the freeze line is written,
unit-tested against mocks, and left unexecuted here (compute policy).

In [3]:
# The execution-order dependency graph (PLAN §0.5) and each phase's compute + status.
print(r"""
   CPU-NOW (here) ── src/ machinery + tests + CPU execution ──────────────┐  ✅ complete, 134 tests green
      rendering ▸ metrics ▸ judge(dry-run) ▸ agreement ▸ BT ▸ MixedLM ▸    │
      power ▸ figures(synthetic)                                          │
                                          │ (all green)                    │
                                          ▼                                │
   FREEZE ─────────────► PREREGISTRATION.md  (git tag prereg-v1) ──────────┘  blocks all GPU work
                                          │
        GPU P0 Pilot ──► P1 Stage-1 gen (4,630) ──► CPU P2 render+metrics ──► API P3 judging
                                          │
        GPU P4 extract/locate ──► GPU P5 dev sweep ─(FREEZE ℓ*,ρ*,variant)─► GPU P6 held-out verify
                                          │                                        │
                                          ▼                                        ▼
                              GPU P7 correspondence (stretch)             P8 evaluation + paper
""")
import pandas as pd
phases = pd.DataFrame([
 ["CPU-now","src/ + tests + CPU execution","CPU","✅ done (here)"],
 ["P0 Pilot","12-gen gate zero","L4","⏳ Colab"],
 ["P1","Stage-1 generation (4,630)","L4","⏳ Colab"],
 ["P2","render + objective metrics","CPU","⏳ (CPU-ready)"],
 ["P3","judging + reliability + BT","API","⏳ Colab/API"],
 ["P4–P7","Stage-2 extract/sweep/verify/stretch","A100","⏳ Colab"],
], columns=["phase","work","compute","status"])
phases.set_index("phase")


   CPU-NOW (here) ── src/ machinery + tests + CPU execution ──────────────┐  ✅ complete, 134 tests green
      rendering ▸ metrics ▸ judge(dry-run) ▸ agreement ▸ BT ▸ MixedLM ▸    │
      power ▸ figures(synthetic)                                          │
                                          │ (all green)                    │
                                          ▼                                │
   FREEZE ─────────────► PREREGISTRATION.md  (git tag prereg-v1) ──────────┘  blocks all GPU work
                                          │
        GPU P0 Pilot ──► P1 Stage-1 gen (4,630) ──► CPU P2 render+metrics ──► API P3 judging
                                          │
        GPU P4 extract/locate ──► GPU P5 dev sweep ─(FREEZE ℓ*,ρ*,variant)─► GPU P6 held-out verify
                                          │                                        │
                                          ▼                                        ▼
                              GPU P7 c

,work,compute,status
phase,,,
CPU-now,src/ + tests + CPU execution,CPU,✅ done (here)
P0 Pilot,12-gen gate zero,L4,⏳ Colab
P1,"Stage-1 generation (4,630)",L4,⏳ Colab
P2,render + objective metrics,CPU,⏳ (CPU-ready)
P3,judging + reliability + BT,API,⏳ Colab/API
P4–P7,Stage-2 extract/sweep/verify/stretch,A100,⏳ Colab


**Hard blocks.** The `prereg-v1` tag blocks all GPU work. The pilot PASS blocks generation. The dev
sweep must **freeze** `(ℓ*, ρ*, variant*)` and git-tag `steer-frozen` **before** any held-out
generation — held-out is touched exactly once, for confirmatory causal verification. This is what
keeps Stage 2 honest: the operating point cannot be tuned on the data it is judged on.

## 0.5 · The repo, and the frozen invariants (checked live)

The study is fully specified before any GPU run. The cell below pulls the invariants **live** from
the frozen YAML configs and asserts them — the same check `scripts/make_configs_check.py` runs — so
this overview can never disagree with the actual experiment.

In [4]:
# Live invariant check against the frozen configs (config.validate_all mirrors PREREG §3/§11).
from p19 import config
rep = config.validate_all()
print("model lock      :", rep["model"])
print("cells           :", rep["cells"], "  (24 distinct; 16-run resolution-V fraction)")
print("prompts / splits:", rep["prompts"]["splits"])
print("steering guards :", rep["steering"])
print("sampling hash   :", rep["hashes"]["sampling"][:16], "...")
assert rep["model"] == "Qwen/Qwen2.5-Coder-7B-Instruct"   # ADR-001 model lock
assert rep["cells"] == {"n": 24, "n_fraction": 16}
print("\nAll frozen invariants hold ✔")

model lock      : Qwen/Qwen2.5-Coder-7B-Instruct
cells           : {'n': 24, 'n_fraction': 16}   (24 distinct; 16-run resolution-V fraction)
prompts / splits: {'dev': 40, 'heldout_seen': 8, 'heldout_unseen': 10}
steering guards : {'kl': 0.3, 'render_min': 0.9}
sampling hash   : e950889836bb1a36 ...

All frozen invariants hold ✔


## 0.6 · The notebook series, and where each figure is born

The eleven notebooks walk the full data-science lifecycle. **Narrative notebooks stay fully executed**
with real outputs; **GPU notebooks stay fully unexecuted** (a per-notebook rule — never a half-run
mix). The synthetic-dry-run notebooks (05, 06a, 07a) run the *real* analysis pipeline on planted data
so you can see exactly what each figure will look like on real data.

In [5]:
# The notebook↔figure map (PLAN §V.3), with execution status in THIS repository.
import pandas as pd
series = pd.DataFrame([
 ["00_overview","executed","the project in one notebook","—"],
 ["01_problem_and_data","executed","RQs; the skill as data + audits; 24-cell table; corpus EDA","—"],
 ["02_oracle_metrics","executed","objective half: render fixtures, metric families, POC, F4","F4, F10"],
 ["03_oracle_preference","executed","preference half: judge, agreement, power, BT","F11"],
 ["04_stage1_generation","GPU scaffold","pilot + factorial generation via vLLM","—"],
 ["05_stage1_analysis","executed (synthetic)","POC→MixedLM→BT→Holm/BH; RQ1–RQ4","F1, F2, F3"],
 ["06a_stage2_locate_extract_synthetic","executed (synthetic)","diff-in-means, probes, patching","F5, F13"],
 ["06b_stage2_locate_extract_gpu","GPU scaffold","real S2.0–S2.2 activation capture","—"],
 ["07a_stage2_steer_verify_synthetic","executed (synthetic)","dose-response, 4-arm verify, correspondence","F6,F7,F8,F9,F12"],
 ["07b_stage2_steer_verify_gpu","GPU scaffold","real S2.3–S2.9 sweep/verify/stretch","—"],
 ["08_results_and_conclusions","executed","outcome-contingent decisions per RQ","(assembles all)"],
], columns=["notebook","status","produces","figures"])
series.set_index("notebook")

,status,produces,figures
notebook,,,
00_overview,executed,the project in one notebook,—
01_problem_and_data,executed,RQs; the skill as data + audits; 24-cell table...,—
02_oracle_metrics,executed,"objective half: render fixtures, metric famili...","F4, F10"
03_oracle_preference,executed,"preference half: judge, agreement, power, BT",F11
04_stage1_generation,GPU scaffold,pilot + factorial generation via vLLM,—
05_stage1_analysis,executed (synthetic),POC→MixedLM→BT→Holm/BH; RQ1–RQ4,"F1, F2, F3"
06a_stage2_locate_extract_synthetic,executed (synthetic),"diff-in-means, probes, patching","F5, F13"
06b_stage2_locate_extract_gpu,GPU scaffold,real S2.0–S2.2 activation capture,—
07a_stage2_steer_verify_synthetic,executed (synthetic),"dose-response, 4-arm verify, correspondence","F6,F7,F8,F9,F12"


**Figures F1–F13** (PLAN §V.1) are each built by a reusable function in `p19.figures` and called from
the notebook that owns them — no figure is hand-drawn. In the executed notebooks the figures are
*previews* on synthetic data with a planted ground truth; on Colab the same functions consume the real
results tables unchanged.

## 0.7 · Current status & how to reproduce

**Status.** The entire CPU-safe stack runs and is green here. No Qwen weights, vLLM, or GPU code is
executed in this environment (compute policy) — those cells are written, mocked/unit-tested,
import-guarded, and left for Colab.

In [6]:
# Reproduce everything CPU-safe from a clean checkout (these are the commands, shown not run).
print("""  # 1. unit tests (all machinery, incl. the synthetic-recovery guarantees)
  python -m pytest -q

  # 2. config invariants (24 cells, res-V fraction, splits, model lock)
  python scripts/make_configs_check.py

  # 3. CPU demos
  python scripts/run_metrics_demo.py      # render the 4 fixtures + full metric vector + POC
  python scripts/run_power_sim.py         # McNemar sizing + BT power at the frozen allocation
  python scripts/run_judge_dryrun.py      # MockJudge both-orders adjudication

  # 4. execute the CPU notebooks top-to-bottom
  jupyter nbconvert --to notebook --execute notebooks/0[0-3]*.ipynb notebooks/05*.ipynb \
                    notebooks/06a*.ipynb notebooks/07a*.ipynb notebooks/08*.ipynb

  # 5. the GPU phase runs on Colab, session-by-session:  see RUNBOOK.md
""")
from pathlib import Path
n_tests = sum(1 for _ in (REPO_ROOT/"tests").glob("test_*.py"))
print(f"test modules: {n_tests}   |   run `python -m pytest -q` for the count (134 passing)")

  # 1. unit tests (all machinery, incl. the synthetic-recovery guarantees)
  python -m pytest -q

  # 2. config invariants (24 cells, res-V fraction, splits, model lock)
  python scripts/make_configs_check.py

  # 3. CPU demos
  python scripts/run_metrics_demo.py      # render the 4 fixtures + full metric vector + POC
  python scripts/run_power_sim.py         # McNemar sizing + BT power at the frozen allocation
  python scripts/run_judge_dryrun.py      # MockJudge both-orders adjudication

  # 4. execute the CPU notebooks top-to-bottom
  jupyter nbconvert --to notebook --execute notebooks/0[0-3]*.ipynb notebooks/05*.ipynb                     notebooks/06a*.ipynb notebooks/07a*.ipynb notebooks/08*.ipynb

  # 5. the GPU phase runs on Colab, session-by-session:  see RUNBOOK.md

test modules: 20   |   run `python -m pytest -q` for the count (134 passing)


---
### How to read the rest of the series
Go in order. **01** frames the problem and shows the skill *as data*. **02–03** build and validate the
two-signal oracle. **04** is the generation scaffold. **05** is the Stage-1 analysis (synthetic
dry-run of the full pipeline). **06–07** are Stage 2 (locate → steer → verify), split into an executed
synthetic half and an unexecuted GPU half. **08** renders every RQ's preregistered decision and writes
the honest-null branches. Governance is `CLAUDE.md`; the authoritative plan is `PLAN.md`; the
derivations are `THEORY.md`; the freeze is `PREREGISTRATION.md`.